#1. Gravação de Áudio com python

In [ ]:
# Instalações e funções de gravação
from IPython.display import Javascript, Audio, display
from google.colab import output
from base64 import b64decode
import os
!pip install -q pydub
from pydub import AudioSegment

language = 'pt'

RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  display(Javascript(RECORD))
  js_result = output.eval_js('record(%d)' % (sec*1000))
  audio = b64decode(js_result.split(',')[1])
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  return f'/content/{file_name}'

print('Ouvindo... (Fale por até 7 segundos)\n')
record_file = record(7) # Você pode aumentar o tempo aqui se quiser
display(Audio(record_file, autoplay=False))

#2. Reconhecimento de Fala com Whisper

In [ ]:
# Transcrição do áudio para texto
!pip install git+https://github.com/openai/whisper.git -q
import whisper

print("Carregando modelo Whisper e transcrevendo...")
model = whisper.load_model("large")
result = model.transcribe(record_file, fp16=False, language=language)
transcription = result["text"]

print("\nVocê disse:")
print(transcription)

#3. Integração com API do Gemini

In [ ]:
# Enviando a transcrição para o Gemini (ATUALIZADA)
!pip install -q google-genai
from google import genai

# Substitua pela sua API Key do Google AI Studio
GOOGLE_API_KEY = "cole sua api-key aqui"

# Inicializando o novo cliente da API
client = genai.Client(api_key=GOOGLE_API_KEY)

# Criando um contexto para a IA agir como recrutadora
contexto_entrevista = """
Você é um recrutador experiente conduzindo uma entrevista de emprego.
Responda de forma profissional, amigável e MUITO concisa (máximo de 2 frases).
Sempre termine sua fala fazendo a próxima pergunta para dar continuidade à entrevista.
"""

# Juntando o contexto com o que você falou
prompt = f"{contexto_entrevista}\n\nO candidato acabou de dizer: '{transcription}'"

print("Gerando resposta do recrutador (Gemini)...")

# Chamada atualizada para o modelo
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt
)

gemini_response = response.text

print("\nRecrutador (Gemini):")
print(gemini_response)

#4. Sintetizando a Resposta como Voz (gTTS)

In [ ]:
# Transformando o texto do Gemini em áudio
!pip install -q gTTS
from gtts import gTTS

print("Gerando áudio da resposta...")
gtts_object = gTTS(text=gemini_response, lang=language, slow=False)
response_audio = "/content/response_audio.wav"
gtts_object.save(response_audio)

display(Audio(response_audio, autoplay=True))